# Training su Google Colab (GPU) — Ispezione Difetti al Microscopio

Questo notebook addestra i modelli sui **dati reali** sfruttando la GPU di Colab,
poi esporta i pesi in `models/` da scaricare e usare in locale.

**Prerequisiti:** Runtime > Cambia tipo di runtime > GPU.

Passi: clona il repo → installa → carica i dati → training → scarica i modelli.

## 1. Clona il repository e installa le dipendenze

In [ ]:
# Sostituisci con l'URL del tuo repo e il branch corretto.
!git clone -b claude/image-inspection-defect-detection-irgeuy https://github.com/daniloflex596/automazioni.git
%cd automazioni/microscopy-defect-inspector
!pip install -q -r requirements-train.txt

## 2. Carica i dati reali

Carica le immagini nelle cartelle:
- `data/good/` — immagini SENZA difetti
- `data/defects/<classe>/` — difettose divise per tipo (cricca, graffio, buco...)
- `data/annotated/` — immagini con cerchio blu + frecce

Puoi usare il pannello File di Colab, `files.upload()`, oppure montare Google Drive.
Per una prova rapida con dati sintetici, esegui la cella seguente.

In [ ]:
# (Opzionale) dataset sintetico per provare la pipeline su Colab
from src.synthetic import generate_dataset
print(generate_dataset())

## 3. Verifica il backend (deve essere 'torch' con GPU disponibile)

In [ ]:
import torch
from src.embeddings import backend_name
print('CUDA disponibile:', torch.cuda.is_available())
print('Backend embedding:', backend_name())

## 4. Training completo (annotazioni → augmentation → anomaly → classificatore)

In [ ]:
from src.train import main
main()  # usa i dati reali se presenti, altrimenti genera i sintetici

## 5. Valutazione rapida (leave-one-out sui difetti)

In [ ]:
from pathlib import Path
from src.config import load_config
from src.classifier import DefectClassifier, _collect_training_samples
import numpy as np

cfg = load_config()
samples = _collect_training_samples(cfg)
correct = 0
for i in range(len(samples)):
    train = samples[:i] + samples[i+1:]
    test_img, test_label = samples[i]
    clf = DefectClassifier(cfg).fit(train)
    pred = clf.predict(test_img)
    correct += int(pred.label == test_label)
print(f'Accuratezza leave-one-out: {correct}/{len(samples)} = {correct/max(1,len(samples)):.2%}')

## 6. Scarica i modelli addestrati

In [ ]:
import shutil
shutil.make_archive('models', 'zip', 'models')
try:
    from google.colab import files
    files.download('models.zip')
except Exception as e:
    print('Scarica manualmente models.zip dal pannello File. Dettaglio:', e)